In [ ]:
!pip install geopandas shapely

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
df = pd.read_csv("/kaggle/input/daily-csv-data-processed-single-2/combined_daily_data.csv")

In [3]:
df.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun,station,name,country,region,latitude,longitude,elevation,timezone
0,2020-01-14,2.5,-1.0,6.0,NaN,NaN,NaN,1.2,NaN,NaN,NaN,KMMU0,Morristown / Birch Hills,US,NJ,40.7993,-74.4149,57,America/New_York
1,2020-01-15,4.8,-1.0,10.0,NaN,NaN,NaN,4.8,NaN,NaN,NaN,KMMU0,Morristown / Birch Hills,US,NJ,40.7993,-74.4149,57,America/New_York
2,2020-01-16,4.5,1.0,8.0,NaN,NaN,318.0,17.3,NaN,NaN,NaN,KMMU0,Morristown / Birch Hills,US,NJ,40.7993,-74.4149,57,America/New_York
3,2020-01-17,-3.8,-7.0,0.0,NaN,NaN,337.0,16.8,NaN,NaN,NaN,KMMU0,Morristown / Birch Hills,US,NJ,40.7993,-74.4149,57,America/New_York
4,2020-01-18,-4.8,-7.0,1.0,NaN,NaN,NaN,5.2,NaN,NaN,NaN,KMMU0,Morristown / Birch Hills,US,NJ,40.7993,-74.4149,57,America/New_York


In [4]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Drop rows where date could not be parsed
df.dropna(subset=['date'], inplace=True)

start_date = '2013-01-01'
end_date = '2024-12-31'

df_filtered = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"Filtered DataFrame shape (2013-2024): {df_filtered.shape}")
print(f"Filtered Date Range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")

Original DataFrame shape: (1074473, 19)
Filtered DataFrame shape (2013-2024): (943875, 19)
Filtered Date Range: 2013-01-01 00:00:00 to 2024-12-31 00:00:00


In [7]:
geometry = [Point(xy) for xy in zip(df_filtered['longitude'], df_filtered['latitude'])]
gdf_points = gpd.GeoDataFrame(df_filtered, geometry=geometry, crs='EPSG:4326')

In [8]:
print("Original DataFrame with Point Geometry:")
print(gdf_points.head())
print("\nCRS of points:", gdf_points.crs)

Original DataFrame with Point Geometry:
        date  tavg  tmin  tmax  prcp  snow   wdir  wspd  wpgt  pres  tsun  \
0 2020-01-14   2.5  -1.0   6.0   NaN   NaN    NaN   1.2   NaN   NaN   NaN   
1 2020-01-15   4.8  -1.0  10.0   NaN   NaN    NaN   4.8   NaN   NaN   NaN   
2 2020-01-16   4.5   1.0   8.0   NaN   NaN  318.0  17.3   NaN   NaN   NaN   
3 2020-01-17  -3.8  -7.0   0.0   NaN   NaN  337.0  16.8   NaN   NaN   NaN   
4 2020-01-18  -4.8  -7.0   1.0   NaN   NaN    NaN   5.2   NaN   NaN   NaN   

  station                      name country region  latitude  longitude  \
0   KMMU0  Morristown / Birch Hills      US     NJ   40.7993   -74.4149   
1   KMMU0  Morristown / Birch Hills      US     NJ   40.7993   -74.4149   
2   KMMU0  Morristown / Birch Hills      US     NJ   40.7993   -74.4149   
3   KMMU0  Morristown / Birch Hills      US     NJ   40.7993   -74.4149   
4   KMMU0  Morristown / Birch Hills      US     NJ   40.7993   -74.4149   

   elevation          timezone                

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [9]:
county_shapefile_path = '/kaggle/input/county-shapes-us-dataset/cb_2023_us_county_500k/cb_2023_us_county_500k.shp'

try:
    gdf_counties = gpd.read_file(county_shapefile_path)
    print(f"\nSuccessfully loaded county shapefile from: {county_shapefile_path}")
    print("County GeoDataFrame Info:")
    
    print("\nCRS of counties:", gdf_counties.crs) # Should be EPSG:4269 or similar NAD83

    if gdf_points.crs != gdf_counties.crs:
        print(f"\nConverting county CRS from {gdf_counties.crs} to {gdf_points.crs}...")
        gdf_counties = gdf_counties.to_crs(gdf_points.crs)
        print("County CRS after conversion:", gdf_counties.crs)

    # Spatial Join --
    print("\nPerforming spatial join...")
    gdf_joined = gpd.sjoin(gdf_points, gdf_counties, how='left', predicate='within')
    print("Spatial join completed.")
    # print("Columns after join:", gdf_joined.columns

    county_col_name = 'NAME'        # County Name
    state_name_col = 'STATE_NAME'   # Full State Name
    fips_code_col = 'GEOID'         # Full County FIPS (STATEFP + COUNTYFP)

    # Check if expected columns exist after join
    if county_col_name not in gdf_joined.columns:
         raise KeyError(f"County name column '{county_col_name}' not found after join.")
    if state_name_col not in gdf_joined.columns:
         raise KeyError(f"State name column '{state_name_col}' not found after join.")
    if fips_code_col not in gdf_joined.columns:
         raise KeyError(f"FIPS code column '{fips_code_col}' not found after join.")

    gdf_joined['county'] = gdf_joined[county_col_name]
    gdf_joined['state'] = gdf_joined[state_name_col]
    gdf_joined['fips'] = gdf_joined[fips_code_col]
    print("County, State, and FIPS columns extracted.")


    # Clean up and Final DataFrame --
    original_cols = list(df.columns)
    # Define the new columns we created
    new_cols = ['county', 'state', 'fips']
    # Define the geometry column 
    geom_col_name = gdf_joined.geometry.name 
    desired_cols = list(dict.fromkeys(original_cols + new_cols))
    df_final = gdf_joined[desired_cols + [geom_col_name]]

    # Check for points that didn't join (resulting in NaN county info)
    no_match_count = df_final['county'].isna().sum()
    if no_match_count > 0:
        print(f"\nWarning: {no_match_count} out of {len(df_final)} points did not fall within any county boundary in the shapefile.")
        # This is expected for points over ocean or outside US territory coverage
except FileNotFoundError:
    print(f"\nERROR: County shapefile not found at '{county_shapefile_path}'.")
    print("Please download the county shapefile (e.g., from US Census TIGER/Line)")
    print("and update the 'county_shapefile_path' variable in the script.")
    df_final = df.copy() # Create a copy to add empty columns
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None # Add placeholder geometry column if needed later

except KeyError as e:
    print(f"\nERROR: A required column name was not found: {e}")
    print("Please check the column names in your shapefile attributes (e.g., using gdf_counties.head())")
    print("and ensure the variables 'county_col_name', 'state_name_col', 'fips_code_col' match.")
    df_final = df.copy()
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    df_final = df.copy()
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None


print("\nFinal GeoDataFrame with County, State, and FIPS:")
print(df_final.head())


Successfully loaded county shapefile from: /kaggle/input/county-shapes-us-dataset/cb_2023_us_county_500k/cb_2023_us_county_500k.shp
County GeoDataFrame Info:

CRS of counties: EPSG:4269

Converting county CRS from EPSG:4269 to EPSG:4326...
County CRS after conversion: EPSG:4326

Performing spatial join...
Spatial join completed.
County, State, and FIPS columns extracted.


Final GeoDataFrame with County, State, and FIPS:
        date  tavg  tmin  tmax  prcp  snow   wdir  wspd  wpgt  pres  ...  \
0 2020-01-14   2.5  -1.0   6.0   NaN   NaN    NaN   1.2   NaN   NaN  ...   
1 2020-01-15   4.8  -1.0  10.0   NaN   NaN    NaN   4.8   NaN   NaN  ...   
2 2020-01-16   4.5   1.0   8.0   NaN   NaN  318.0  17.3   NaN   NaN  ...   
3 2020-01-17  -3.8  -7.0   0.0   NaN   NaN  337.0  16.8   NaN   NaN  ...   
4 2020-01-18  -4.8  -7.0   1.0   NaN   NaN    NaN   5.2   NaN   NaN  ...   

   country region latitude longitude elevation          timezone  county  \
0       US     NJ  40.7993  -74.4149     

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [10]:
df_final.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,...,country,region,latitude,longitude,elevation,timezone,county,state,fips,geometry
0,2020-01-14,2.5,-1.0,6.0,NaN,NaN,NaN,1.2,NaN,NaN,...,US,NJ,40.7993,-74.4149,57,America/New_York,Morris,New Jersey,34027,POINT (-74.41490 40.79930)
1,2020-01-15,4.8,-1.0,10.0,NaN,NaN,NaN,4.8,NaN,NaN,...,US,NJ,40.7993,-74.4149,57,America/New_York,Morris,New Jersey,34027,POINT (-74.41490 40.79930)
2,2020-01-16,4.5,1.0,8.0,NaN,NaN,318.0,17.3,NaN,NaN,...,US,NJ,40.7993,-74.4149,57,America/New_York,Morris,New Jersey,34027,POINT (-74.41490 40.79930)
3,2020-01-17,-3.8,-7.0,0.0,NaN,NaN,337.0,16.8,NaN,NaN,...,US,NJ,40.7993,-74.4149,57,America/New_York,Morris,New Jersey,34027,POINT (-74.41490 40.79930)
4,2020-01-18,-4.8,-7.0,1.0,NaN,NaN,NaN,5.2,NaN,NaN,...,US,NJ,40.7993,-74.4149,57,America/New_York,Morris,New Jersey,34027,POINT (-74.41490 40.79930)


In [11]:
df_final = df_final.drop(["geometry"], axis=1)

In [12]:
df_final.to_csv("wather_data_2014_24_county_info_daily.csv", index=False)

In [13]:
import gc
gc.collect()

del df_final, gdf_points, df, geometry

In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

df = pd.read_csv("/kaggle/input/hourly-csv-data-processed-single-2/combined_hourly_data.csv")

df['date'] = pd.to_datetime(df['date'], errors='coerce')

df.dropna(subset=['date'], inplace=True)

# date range boundaries 
start_date = '2013-01-01'
end_date = '2024-12-31'

df_filtered = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"Filtered DataFrame shape (2013-2024): {df_filtered.shape}")
print(f"Filtered Date Range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")

/tmp/ipykernel_103/2859041661.py:5: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/hourly-csv-data-processed-single-2/combined_hourly_data.csv")


Original DataFrame shape: (25413657, 21)
Filtered DataFrame shape (2013-2024): (23298781, 21)
Filtered Date Range: 2013-01-01 00:00:00 to 2024-12-31 00:00:00


In [ ]:
geometry = [Point(xy) for xy in zip(df_filtered['longitude'], df_filtered['latitude'])]
gdf_points = gpd.GeoDataFrame(df_filtered, geometry=geometry, crs='EPSG:4326')
print("Original DataFrame with Point Geometry:")
print(gdf_points.head())
print("\nCRS of points:", gdf_points.crs)

county_shapefile_path = '/kaggle/input/county-shapes-us-dataset/cb_2023_us_county_500k/cb_2023_us_county_500k.shp'

try:
    gdf_counties = gpd.read_file(county_shapefile_path)
    print(f"\nSuccessfully loaded county shapefile from: {county_shapefile_path}")
    print("County GeoDataFrame Info:")
    print("\nCRS of counties:", gdf_counties.crs) # Should be EPSG:4269 or similar NAD83

    # -- 4. Ensure CRS Match --
    if gdf_points.crs != gdf_counties.crs:
        print(f"\nConverting county CRS from {gdf_counties.crs} to {gdf_points.crs}...")
        gdf_counties = gdf_counties.to_crs(gdf_points.crs)
        print("County CRS after conversion:", gdf_counties.crs)

    # Spatial Join --
    print("\nPerforming spatial join...")
    gdf_joined = gpd.sjoin(gdf_points, gdf_counties, how='left', predicate='within')
    print("Spatial join completed.")
    county_col_name = 'NAME'        # County Name
    state_name_col = 'STATE_NAME'   # Full State Name
    fips_code_col = 'GEOID'         # Full County FIPS (STATEFP + COUNTYFP)

    # Check if expected columns exist after join
    if county_col_name not in gdf_joined.columns:
         raise KeyError(f"County name column '{county_col_name}' not found after join.")
    if state_name_col not in gdf_joined.columns:
         raise KeyError(f"State name column '{state_name_col}' not found after join.")
    if fips_code_col not in gdf_joined.columns:
         raise KeyError(f"FIPS code column '{fips_code_col}' not found after join.")

    gdf_joined['county'] = gdf_joined[county_col_name]
    gdf_joined['state'] = gdf_joined[state_name_col]
    gdf_joined['fips'] = gdf_joined[fips_code_col]
    print("County, State, and FIPS columns extracted.")

    # The columns from the original DataFrame we want to keep
    original_cols = list(df.columns)
    # Define the new columns we created
    new_cols = ['county', 'state', 'fips']
    geom_col_name = gdf_joined.geometry.name

    desired_cols = list(dict.fromkeys(original_cols + new_cols))
    df_final = gdf_joined[desired_cols + [geom_col_name]]

    # Check for points that didn't join (resulting in NaN county info)
    no_match_count = df_final['county'].isna().sum()
    if no_match_count > 0:
        print(f"\nWarning: {no_match_count} out of {len(df_final)} points did not fall within any county boundary in the shapefile.")
        # This is expected for points over ocean or outside US territory coverage

except FileNotFoundError:
    print(f"\nERROR: County shapefile not found at '{county_shapefile_path}'.")
    print("Please download the county shapefile (e.g., from US Census TIGER/Line)")
    print("and update the 'county_shapefile_path' variable in the script.")
    df_final = df.copy() # Create a copy to add empty columns
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None 

except KeyError as e:
    print(f"\nERROR: A required column name was not found: {e}")
    print("Please check the column names in your shapefile attributes (e.g., using gdf_counties.head())")
    print("and ensure the variables 'county_col_name', 'state_name_col', 'fips_code_col' match.")
    df_final = df.copy()
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    df_final = df.copy()
    df_final['county'] = pd.NA
    df_final['state'] = pd.NA
    df_final['fips'] = pd.NA
    df_final['geometry'] = None


print("\nFinal GeoDataFrame with County, State, and FIPS:")
print(df_final.head())

Original DataFrame with Point Geometry:
            date  hour  temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt  ...  \
15023 2013-01-01     3   1.0  -4.1  69.0   NaN   NaN  230.0  20.5   NaN  ...   
15024 2013-01-01     4   2.0  -4.1  64.0   NaN   NaN  230.0  14.8   NaN  ...   
15025 2013-01-01     5   3.0  -2.9  65.0   NaN   NaN  240.0  18.4   NaN  ...   
15026 2013-01-01     6   3.0  -1.9  70.0   NaN   NaN  250.0  13.0   NaN  ...   
15027 2013-01-01     7   3.0  -1.9  70.0   NaN   NaN  260.0   9.4   NaN  ...   

       coco  station                      name country region latitude  \
15023   NaN    KMMU0  Morristown / Birch Hills      US     NJ  40.7993   
15024   NaN    KMMU0  Morristown / Birch Hills      US     NJ  40.7993   
15025   NaN    KMMU0  Morristown / Birch Hills      US     NJ  40.7993   
15026   NaN    KMMU0  Morristown / Birch Hills      US     NJ  40.7993   
15027   NaN    KMMU0  Morristown / Birch Hills      US     NJ  40.7993   

      longitude  elevation        

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()



Successfully loaded county shapefile from: /kaggle/input/county-shapes-us-dataset/cb_2023_us_county_500k/cb_2023_us_county_500k.shp
County GeoDataFrame Info:

CRS of counties: EPSG:4269

Converting county CRS from EPSG:4269 to EPSG:4326...
County CRS after conversion: EPSG:4326

Performing spatial join...


In [ ]:
df_final.head()

In [ ]:
df_final = df_final.drop(["geometry"], axis=1)

In [ ]:
df_final.to_csv("wather_data_2014_24_county_info_hourly.csv", index=False)

In [ ]:
print("DONE")